In [ ]:
import sys
from pathlib import Path
import polars as pl

sys.path.append(str(Path("..").resolve()))
from fleetsense.features.data_loader import get_dataset, FEATURES, get_features_and_target
from fleetsense.monitoring.distribution_monitoring import monitor_all_features, build_baselines, check_drift
from fleetsense.monitoring.plotting import (
    plot_psi_heatmap,
    plot_psi_timeseries,
    rank_features_by_drift,
)
from fleetsense.model.base_model import load_baseline_model

In [ ]:
df = get_dataset()
df = df.with_columns(
    pl.col("week_start")
    .str.to_datetime("%Y-%m-%dT%H:%M:%S%.f", strict=False)  # skip if week_start is already a Date/Datetime
    .dt.truncate("1mo")
    .alias("month_start")
)

In [ ]:
model = load_baseline_model()
X_all, _ = get_features_and_target(df)
df = df.with_columns(pl.Series("predicted_ship_type", model.predict(X_all)))

In [ ]:
baselines = build_baselines(
    df,
    FEATURES,
    "ship_type",
    predicted_class_col="predicted_ship_type",
)

In [ ]:
results = monitor_all_features(
    baselines, df, FEATURES, "month_start", class_col="ship_type", predicted_class_col="predicted_ship_type"
)

In [ ]:
results

In [ ]:
flagged = check_drift(results)
flagged.filter(pl.col("feature") == "__predicted_class_balance__")

In [ ]:
for row in flagged.iter_rows(named=True):
    print(f"{row['period']}: {row['feature']}, {row['ship_type']}, {row['psi']}")

In [ ]:
plot = plot_psi_heatmap(results, class_col="ship_type", figsize_per_panel=(10, 10))

In [ ]:
plot_psi_timeseries(results, "frac_time_slow", "ship_type")

In [ ]:
rank_features_by_drift(results, "ship_type")